In [4]:
import os
import re
from dotenv import load_dotenv
load_dotenv()

from langchain_gigachat.chat_models import GigaChat

# Инициализация GigaChat
llm = GigaChat(
    credentials=os.getenv("GIGA_KEY"),
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000
)

print("✅ GigaChat подключен!")
print("Тест:", llm.invoke("2+2=").content)

/Users/egorlubar/PyCharmMiscProject/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ GigaChat подключен!
Тест: $2 + 2 = 4$


In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Промпт для извлечения количества человек
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Ты — аналитик заявок на аренду жилья. Извлеки ТОЧНОЕ количество проживающих.

Правила:
- "пара", "двое", "молодая семья" = 2
- "семья с N детьми" = 2 + N
- "один", "на одного" = 1
- "2 семьи по 3 человека" = 6
- Числительные ("троих", "четверых") переводи в цифры

Верни ТОЛЬКО целое число.

Заявка: {text}
Количество:"""
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

print("✅ Цепочка создана!")

✅ Цепочка создана!


In [6]:
# Тестовые заявки из твоего файла
test_texts = [
    "ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1 двухместный номер, 1 трехместный, недорого. или как вариант дом на 2 семьи (5 чел)",
    "2 семьи по 3 человека (2 взрослых и ребенок) с 01.09-13.09 снимем жильё в пределах 5 минут от пляжа, 1500 руб/сутки. Рассмотрим все варианты.",
    "Добрый день! Ищем жильё (гостевой дом) с парковкой в Лазаревском, планируем приезд 5-6 августа на 7 дней, двое взрослых и двое детей (16 и 2 г) не далеко от моря. Рассмотрим ваши предложения!",
    "Здравствуйте ! Ищем жилье в Лазаревском 4 человека, с бассейном с 1.08 по 12.08!",
    "Ищем жилье в самом Лазаревском для двоих, 2-местный номер с удобствами, с 7-8 сентября не менее чем на неделю, 700 р. за номер",
    "Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от пляжа",
    "Ищем жильё для 2-х взрослых, ребёнок (7 лет) и собака. Только Чёрное море, не Крым и Анапа. С 15 июля на 10 дней.",
    "Интересует жилье в Сочи у пляжа Ривьера (максимально близко к морю!!!) с 8 по 15 июля. 1 номер/квартира/комната, 3 человека. Предложения в л/с",
    "Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослых с 5.09 интересует. Только в личку.",
    "Здравствуйте, ищем жилье с 10.08по 20.08,два взрослых,два ребенка,не далеко от моря.",
    "Ищем жилье в Лазаревском с 12 по 18 августа. Не дороже 1000р за комнату. Нас двое.",
    "Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 августа, недалеко от моря. Писать в личку",
    "Здравствуйте🤗 Ищем жильё в центре рядом с морем🌊 Цена за номер до2️⃣0️⃣0️⃣0️руб с удобствами  Приезжаем парой с 10 августа по 18 августа 📆",
    "Здравствуйте!Ищем жильё в Лазаревское.2 взрослых и 1 ребёнок 10 лет.В период с 25.07. По 06.08.",
    "ищем жилье с 8-20 августа с бассейном 3мест номер ,до моря не дольше 10мин, не в гору"
]

print("🧪 ТЕСТ БАЗОВОЙ ЦЕПОЧКИ:")
print("=" * 70)

for i, t in enumerate(test_texts, 1):
    try:
        raw = chain.invoke({"text": t})
        nums = re.findall(r'\d+', str(raw))
        cleaned = nums[0] if nums else "❌"
        print(f"{i:2d}. {cleaned} ← {t[:60]}...")
    except Exception as e:
        print(f"{i:2d}. ❌ ОШИБКА: {e}")

print("=" * 70)

🧪 ТЕСТ БАЗОВОЙ ЦЕПОЧКИ:
 1. 6 ← ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1...
 2. 6 ← 2 семьи по 3 человека (2 взрослых и ребенок) с 01.09-13.09 с...
 3. 4 ← Добрый день! Ищем жильё (гостевой дом) с парковкой в Лазарев...
 4. 4 ← Здравствуйте ! Ищем жилье в Лазаревском 4 человека, с бассей...
 5. 2 ← Ищем жилье в самом Лазаревском для двоих, 2-местный номер с ...
 6. 1 ← Интересует жилье на одного человека с 08.07. до 15.07. Не да...
 7. 3 ← Ищем жильё для 2-х взрослых, ребёнок (7 лет) и собака. Тольк...
 8. 3 ← Интересует жилье в Сочи у пляжа Ривьера (максимально близко ...
 9. 4 ← Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослы...
10. 4 ← Здравствуйте, ищем жилье с 10.08по 20.08,два взрослых,два ре...
11. 2 ← Ищем жилье в Лазаревском с 12 по 18 августа. Не дороже 1000р...
12. 2 ← Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 август...
13. 2 ← Здравствуйте🤗 Ищем жильё в центре рядом с морем🌊 Цена за ном...
14. 3 ← Здравствуйте!Ищем жильё в Лазаре

In [7]:
import pandas as pd

print("=" * 70)
print("📊 ШАГ 3: ОЦЕНКА ТОЧНОСТИ")
print("=" * 70)

# Загрузка данных
print("\n1️⃣ Загрузка данных...")
df = pd.read_csv('rental_12.csv', sep=';')
print(f"✅ Загружено {len(df)} заявок")

# Обработка через модель
print("\n2️⃣ Обработка заявок через LLM...")
print("-" * 70)

results = []
for idx, row in df.iterrows():
    text = row["text"]
    try:
        raw = chain.invoke({"text": text})
        nums = re.findall(r'\d+', str(raw))
        result = nums[0] if nums else "0"
        results.append(result)
        print(f"{idx+1:2d}. {result} ← {text[:50]}...")
    except Exception as e:
        results.append("0")
        print(f"{idx+1:2d}. ❌ ОШИБКА: {e}")

print("-" * 70)

# Сохранение результатов
df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
print(f"\n3️⃣ Результаты сохранены в rental_with_results.csv")

# Расчет точности
print("\n4️⃣ РАСЧЕТ ТОЧНОСТИ:")
print("=" * 70)

df["result_num"] = pd.to_numeric(df["result"], errors='coerce')
df["amount"] = pd.to_numeric(df["amount"], errors='coerce')

correct = (df["result_num"] == df["amount"]).sum()
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"\n📈 РЕЗУЛЬТАТЫ:")
print(f"✅ Правильно: {correct}")
print(f"❌ Ошибок: {errors}")
print(f"🎯 Точность: {accuracy:.1%}")

# Показ ошибок
if errors > 0:
    print(f"\n❌ ОШИБКИ:")
    errors_df = df[df["result_num"] != df["amount"]]
    for idx, row in errors_df.iterrows():
        print(f"Текст: {row['text'][:60]}...")
        print(f"   Правильно: {int(row['amount'])}, LLM: {row['result']}")

print("=" * 70)
print("✅ ГОТОВО!")

📊 ШАГ 3: ОЦЕНКА ТОЧНОСТИ

1️⃣ Загрузка данных...
✅ Загружено 15 заявок

2️⃣ Обработка заявок через LLM...
----------------------------------------------------------------------
 1. 1 ← ищем жилье в центре недалеко от моря с 23.07-03.08...
 2. 6 ← 2 семьи по 3 человека (2 взрослых и ребенок) с 01....
 3. 4 ← Добрый день! Ищем жильё (гостевой дом) с парковкой...
 4. 4 ← Здравствуйте ! Ищем жилье в Лазаревском 4 человека...
 5. 2 ← Ищем жилье в самом Лазаревском для двоих, 2-местны...
 6. 1 ← Интересует жилье на одного человека с 08.07. до 15...
 7. 3 ← Ищем жильё для 2-х взрослых, ребёнок (7 лет) и соб...
 8. 3 ← Интересует жилье в Сочи у пляжа "Ривьера" (максима...
 9. 4 ← Лазаревское, Солоники - бюджетное жилье на 3 суток...
10. 4 ← Здравствуйте, ищем жилье с 10.08по 20.08,два взрос...
11. 2 ← Ищем жилье в Лазаревском с 12 по 18 августа. 
Не д...
12. 2 ← Ищем жильё эконом класса, до 1000 на двоих, с 7 по...
13. 2 ← Здравствуйте🤗
Ищем жильё в центре рядом с морем🌊
Ц...
14. 3 ← Здравству